<a href="https://colab.research.google.com/github/bridgetagboyie16-eng/Photo-to-art-ai-app/blob/main/PhotoToArt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python Pillow gradio

In [10]:
import cv2
import numpy as np
from PIL import Image

def extract_lineart(input_image):
    """
    Clean contour extractor: removes texture noise while keeping
    eyes, nose, lips, jawline, and silhouette perfectly sharp.
    """
    resized_img = input_image.resize((512, 512))
    img_array = np.array(resized_img)

    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

    # Heavier blur eliminates skin pore and shirt fabric noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Balanced thresholds to avoid harsh wrinkled lines
    edges = cv2.Canny(blurred, threshold1=60, threshold2=140)

    # Thicken lines slightly for clean AI interpretation
    kernel = np.ones((2, 2), np.uint8)
    edges = cv2.dilate(edges, kernel, iterations=1)

    edges_3channel = np.stack([edges] * 3, axis=-1)
    return Image.fromarray(edges_3channel)

print("✅ Clean Contour Extractor ready!")

✅ Clean Contour Extractor ready!


In [4]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

print("⏳ 1/3 Downloading the Stencil Assistant (ControlNet Canny)...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16
)

print("⏳ 2/3 Downloading the Master Painter (Stable Diffusion 1.5)...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
)

print("⏳ 3/3 Optimizing speed and sending to the GPU...")
# UniPCMultistepScheduler makes the AI paint in 20 crisp steps instead of 50
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

# Move the whole model onto Google's free T4 GPU
pipe = pipe.to("cuda")

# Low-memory optimization so your session never crashes
pipe.enable_attention_slicing()

print("✅ The AI Brain is fully loaded and ready to paint!")

⏳ 1/3 Downloading the Stencil Assistant (ControlNet Canny)...
⏳ 2/3 Downloading the Master Painter (Stable Diffusion 1.5)...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

⏳ 3/3 Optimizing speed and sending to the GPU...
✅ The AI Brain is fully loaded and ready to paint!


In [ ]:
import gradio as gr

# Specialized prompt engineered specifically for fine graphite portrait art
GRAPHITE_PROMPT = (
    "masterpiece portrait drawing, fine graphite pencil sketch, delicate pencil shading, "
    "blended graphite tones, cross-hatching, smooth skin rendering, clean fine art paper texture, "
    "hand drawn by a master portrait artist, 8k resolution, authentic pencil strokes"
)

NEGATIVE_PROMPT = (
    "ugly, harsh black outlines, deformed, wrinkle lines, dark shadows, heavy contrast, "
    "photographic, color, digital painting, smooth plastic, cartoon, 3d render, horror, bad eyes"
)

def generate_artwork(input_image):
    if input_image is None:
        return None, None

    # Step A: Clean outline stencil
    edge_stencil = extract_lineart(input_image)

    # Step B: Generate artwork
    # controlnet_conditioning_scale=0.9 gives the AI room to create soft graphite shading
    output = pipe(
        prompt=GRAPHITE_PROMPT,
        negative_prompt=NEGATIVE_PROMPT,
        image=edge_stencil,
        num_inference_steps=28,
        guidance_scale=7.5,
        controlnet_conditioning_scale=0.9
    ).images[0]

    return edge_stencil, output

with gr.Blocks(title="Photo-to-Art AI") as app:
    gr.Markdown("# 🎨 Fine Graphite Pencil Studio")
    gr.Markdown("Transform your portrait into fine hand-drawn graphite artwork.")

    with gr.Row():
        with gr.Column():
            input_box = gr.Image(type="pil", label="Upload Portrait Photo")
            generate_btn = gr.Button("✨ Create Graphite Portrait", variant="primary")

        with gr.Column():
            stencil_display = gr.Image(type="pil", label="Detected Contours")
            output_display = gr.Image(type="pil", label="Graphite Drawing Output")

    generate_btn.click(
        fn=generate_artwork,
        inputs=[input_box],
        outputs=[stencil_display, output_display]
    )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b3ef5dfa7af5253a19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
